In [ ]:
!pip install torch -q
!pip install transformers -q
!pip install numpy -q
!pip install langchain -q
!pip install langchain_community -q
!pip install langchain-chroma -q
!pip install sentence_transformers -q
!pip install -U langchain-huggingface -q
!pip install -U langchain-huggingface huggingface_hub

In [ ]:
!pip install -U langchain langchain-core langchain-huggingface huggingface_hub

In [ ]:
## Initialize HuggingFace LLM

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
import os

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

llm = HuggingFaceEndpoint(
    repo_id  = "mistralai/Mistral-7B-v0.1",
    temperature=0.5,
    max_new_tokens=500,
    huggingfacehub_api_token = "HF_TOKEN"
)

In [ ]:
## Initialize Embedding Model

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
## Initialize Output Parser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()


In [ ]:
## Load PDF Document

In [ ]:
!pip install pypdf -qU

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = "/content/Content"

docs = []
for filename in sorted(os.listdir(pdf_folder)):
    if filename.lower().endswith(".pdf"):
        filepath = os.path.join(pdf_folder, filename)
        loader = PyPDFLoader(filepath)
        paper_docs = loader.load()
        paper_title = os.path.splitext(filename)[0]
        for doc in paper_docs:
            doc.metadata["paper_title"] = paper_title
        docs.extend(paper_docs)
        print(f"Loaded {len(paper_docs)} pages from {filename}")

In [ ]:
len(docs)

In [ ]:
## Split documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(len(splits))

In [ ]:
## Create Vector Store and Retriever

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
## Define Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Answer this question using the provided context only. When you use information
from a specific source, reference it using its [n] number from the context.

{question}

Context:
{context}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
## Chain Retriever and Prompt Template with LLM (citation-grounded)

In [ ]:
def format_docs_with_sources(docs):
    formatted = []
    for i, doc in enumerate(docs):
        paper = doc.metadata.get("paper_title", "unknown paper")
        page = doc.metadata.get("page", "unknown")
        formatted.append(f"[{i+1}] (Paper: {paper}, page {page})\n{doc.page_content}")
    return "\n\n".join(formatted)

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

rag_chain_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=(
        {
            "context": lambda x: format_docs_with_sources(x["context"]),
            "question": lambda x: x["question"],
        }
        | prompt
        | llm
        | output_parser
    )
)

def ask(question):
    result = rag_chain_with_sources.invoke(question)
    print("Answer:\n")
    print(result["answer"])
    print("\nSources:")
    seen = set()
    for doc in result["context"]:
        key = (doc.metadata.get("paper_title", "unknown paper"), doc.metadata.get("page", "?"))
        if key not in seen:
            seen.add(key)
            print(f"- {key[0]}, page {key[1]}")
    return result

In [ ]:
## Invoke RAG Chain with Example Questions

In [ ]:
_ = ask("What is Agentic AI?")

In [ ]:
_ = ask("What are the latest findings in Agentic AI?")

In [ ]:
_ = ask("What are the future findings expected in Agentic AI?")